In [ ]:
# Create and validate our Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("projet-final") \
        .master("local[*]") \
        .getOrCreate()

print("Spark Version:", spark.version)

In [ ]:
# Test the connection to the HDFS
hdfs_root_path = "hdfs://namenode:9000/user/mti850/processed/"

from pyspark.sql import functions as sf
from pyspark.sql.types import StringType

df_orders = spark.read.parquet(f"{hdfs_root_path}olist_orders_dataset.parquet")
df_customers = spark.read.parquet(f"{hdfs_root_path}olist_customers_dataset.parquet")

df_orders.show(1)
df_customers.show(1)

In [ ]:
# Calculate total_purchases for each client
df_purchases_by_client = (df_orders.join(df_customers, "customer_id")
                             .where(sf.col("order_status") == "delivered")
                             .groupBy(df_customers["customer_unique_id"])
                             .agg(sf.count(df_customers["customer_id"])
                                  .alias("total_purchases"))
                             .orderBy(sf.desc("total_purchases"))
                         )

df_purchases_by_client.show(5)

In [ ]:
# How many client have more than one purchase?
print(df_purchases_by_client.filter(sf.col("total_purchases") > 1).count())
print(df_purchases_by_client.filter(sf.col("total_purchases") <= 1).count())

Clients avec reachat : 2 801 \
Clients a achat unique : 90 557 \
Total de clients : 93 358

**Taux de reachat reel : 2 801 / 93 358 = 3.00 %**

#### Avoiding Data Leakage

If we give the model the statistics calculated over both years, the model "cheats" by reading the future. In production, a model never knows the future.

Your idea of ​​cutting time into blocks of time is called Time-Based Windowing in data science.

***“As soon as a customer receives and reviews their very first order, can we immediately predict whether they will repurchase on their own within 90 days, or whether they may never return?”***

#### Beware of Right Censoring:
Olist's data ends at the end of August 2018. If a customer places their first order on August 20, 2018, only 10 days have passed before the data set ends. We cannot say that he would not have redeemed within 90 days.

To be scientifically rigorous, we exclude the first orders placed in the last 90 days of the dataset (after May 2018).

In [ ]:
# Number and order client's order by chronogical order
from pyspark.sql.window import Window

# Table's cleaning
df_customer_orders = (df_orders.join(df_customers, "customer_id")
                        .where(sf.col("order_status") == "delivered")
                     )

# Define the window's rule
window_client = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp")

# Apply enum adding a new column
df_orders_ranked = df_customer_orders.withColumn(
    "order_seq", sf.row_number().over(window_client)
)

df_orders_ranked.show(3)

In [ ]:
# Find next_order_date for each order
df_with_next_date = df_orders_ranked.withColumn(
    "next_order_date", sf.lead("order_purchase_timestamp", 1).over(window_client)
)

(df_with_next_date.filter(
    sf.col("next_order_date").isNotNull())
    .select("customer_unique_id", "order_seq", "order_purchase_timestamp", "next_order_date")
    .show(5)
)

In [ ]:
# Calculate how long it took to each customer to purchase again
df_diff_purchase_dates = df_with_next_date.withColumn(
    "days_to_next_order", sf.datediff("next_order_date", "order_purchase_timestamp"),
)

# Set if did_purchase_again in 90 days or less
df_did_purchase_again = df_diff_purchase_dates.withColumn(
    "did_purchase_again",
    sf.when(
        (sf.col("days_to_next_order").isNotNull())
        & (sf.col("days_to_next_order") <= 90),
        1,
    ).otherwise(0),
)

df_did_purchase_again = df_did_purchase_again.cache()

df_did_purchase_again.show(3)

# Filter only first orders to avoid data leakage 
df_first_orders = df_did_purchase_again.filter(sf.col("order_seq") == 1)

df_first_orders.groupBy("did_purchase_again").count().show()

In [ ]:
# Right-censoring: To ensure our model learns from unbiased observations, 
# every customer included in the training set must have had at least 90 full days 
# of opportunity to make a purchase.

most_recent_order_timestamp = (df_first_orders
                   .select(sf.max(sf.col("order_purchase_timestamp")).alias("most_recent_order_timestamp"))
                   )

most_recent_order_timestamp.show()

right_censored_timestamp = (most_recent_order_timestamp
                            .select('most_recent_order_timestamp',
                                       sf.date_sub(
                                           most_recent_order_timestamp["most_recent_order_timestamp"], 
                                           90).cast('timestamp').alias('right_censored_timestamp')
                                       )
                           )

right_censored_timestamp.show()

cutoff_date = right_censored_timestamp.first()["right_censored_timestamp"]
print("Date limite (cutoff) :", cutoff_date)

In [ ]:
# Filter first orders that don't exceed the limit date
df_filtered_first_orders = df_first_orders.filter(sf.col("order_purchase_timestamp") < cutoff_date)

df_filtered_first_orders.show(3)
df_filtered_first_orders.groupBy("did_purchase_again").count().show()


### Possibles candidates for feature ingeniering

olist_order_items_dataset: shipping_limit_date, freight_value

olist_order_payments_dataset: payment_value

olist_order_reviews_dataset: review_score

olist_orders_dataset: order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date

In [ ]:
# Transforming raw dates into usable numerical variables

# Delivery delay times and actual delivery days
df_filtered_first_orders = (
    df_filtered_first_orders
    # Delay day (positive) : delivery_date - purchase_date
    .withColumn(
        "actual_delivery_days", 
        sf.datediff(
            sf.col("order_delivered_customer_date"),
            sf.col("order_purchase_timestamp")
        )
    )
    # Delay days: actual_delivery_date - estimated_delivery_date
    .withColumn(
        "delay_days", 
        sf.datediff(
            sf.col("order_delivered_customer_date"),
            sf.col("order_estimated_delivery_date")
        )
    )
    # Binary : 1 delayed, 0 on time
    .withColumn(
        "is_delayed",
        sf.when(sf.col("delay_days") > 0, 1).otherwise(0)
    )
)                    

df_filtered_first_orders.show(3)

In [ ]:
# Read order items table
df_order_items = spark.read.parquet(f"{hdfs_root_path}olist_order_items_dataset.parquet")
df_order_items.show(3)

df_order_totals = (
    df_order_items.groupBy("order_id").agg(
            sf.count("order_item_id").alias("total_items"),
            sf.sum("price").alias("total_price"),
            sf.sum("freight_value").alias("shipping_cost")
        )
)

# Proportion of shipping costs relative to the value of the shopping cart.
df_order_totals = df_order_totals.withColumn(
    "freight_ratio",
    sf.round(sf.col("shipping_cost") / (sf.col("total_price") + sf.col("shipping_cost")), 4)
)

df_order_totals.show(3)

In [ ]:
# Read payments table
df_order_payments = spark.read.parquet(f"{hdfs_root_path}olist_order_payments_dataset.parquet")
df_order_payments.show(3)

df_payment_totals = (
    df_order_payments.groupBy("order_id").agg(
            sf.sum("payment_value").alias("total_payment"),
            sf.max("payment_installments").alias("max_payment_installments"),
            sf.countDistinct("payment_type").alias("total_payment_types")
        )
)

df_payment_totals.show(3)

In [ ]:
# Read order reviews table
df_order_reviews = spark.read.parquet(f"{hdfs_root_path}olist_order_reviews_dataset.parquet")
df_order_reviews.show(3)

df_review_totals = (
    df_order_reviews.groupBy("order_id").agg(
            sf.count("review_id").alias("total_reviews"),
            sf.avg("review_score").alias("avg_review_score"),
        )
)

df_review_totals.show(3)

In [ ]:
## Left join all tables to the order table
df_ml_dataset = (
    df_filtered_first_orders
        .join(df_order_totals, "order_id", "left")
        .join(df_payment_totals, "order_id", "left")
        .join(df_review_totals, "order_id", "left")
                
)

df_ml_dataset.show(1)

print("Nombre de lignes attendu : 75260")
print("Nombre de lignes obtenu  :", df_ml_dataset.count())

In [ ]:
# Machine learning doesn't accept NULL values.
# To know which columns will need a null treatment we diagnose the columns
feature_cols = [
    "actual_delivery_days", "delay_days", "is_delayed",
    "total_items", "total_price", "shipping_cost", "freight_ratio",
    "total_payment", "max_payment_installments", "total_payment_types",
    "total_reviews", "avg_review_score"
]

df_ml_dataset.select([
    sf.count(sf.when(sf.col(c).isNull(), c)).alias(c) for c in feature_cols
]).show(vertical=True)

In [ ]:
# Replace the total_reviews nulls by zero
df_ml_dataset = df_ml_dataset.fillna({"total_reviews": 0})

# Replace the avg_review_score nulls by the average because if we set it to zero 
# the model will think that the note was the lowest and that's not the case.
df_ml_dataset = (df_ml_dataset
    .fillna({"avg_review_score": df_ml_dataset.agg(sf.avg("avg_review_score")).first()[0]}))

# Filter the three orders that don't have a delivery_date/payments
df_ml_dataset = df_ml_dataset.na.drop(subset=("total_payment", "actual_delivery_days"))

df_ml_dataset.select([
    sf.count(sf.when(sf.col(c).isNull(), c)).alias(c) for c in feature_cols
]).show(vertical=True)

print("Nombre de lignes obtenu  :", df_ml_dataset.count())

In [ ]:
# Save the dataframe in HDFS
analytics_path = "hdfs://namenode:9000/user/mti850/analytics/first_orders_features.parquet"

df_ml_dataset.write.mode("overwrite").parquet(analytics_path)

In [ ]:
spark.stop()